# SAE Steering – Results Sanity Check

Loads the analytics JSON downloaded from the admin *Results → Download analytics (JSON)* button and verifies it can be turned into usable DataFrames.

**Expected input shape** (produced by `fetch_results` in `server/plugins/sae_steering/__init__.py`):

```
{
  "study_guid": str,
  "sample": {...},
  "approaches": {...},
  "questionnaire": {...},
  "moderators": {...},
  "insights": [...],
  "participants":        [ …detailed per-participant dicts… ],
  "participants_table":  [ …admin-table rows (one per participation)… ]
}
```

Steps below:
1. Load the JSON, peek at the top-level structure.
2. (optional) Drop pre-launch / dev runs via a time window on `time_finished`.
3. Turn `participants_table` into a DataFrame (the same one the admin UI renders).
4. Explode `participants[*].phase_movie_feedback` into a long-form feedback DataFrame.
5. Explode `participants[*].phase_questionnaires` into a questionnaire DataFrame.
6. A few quick aggregates you'd want to double-check before paying people.

In [18]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Point this at whichever analytics dump you want to inspect.  The file is the
# one the admin "Download analytics (JSON)" button produces.
RESULTS_PATH = Path.home() / "Downloads" / "results_JsbnDzlvW7kWgGngu0fMwQkaQY45ozgC-2.json"

with RESULTS_PATH.open("r") as fh:
    data = json.load(fh)

print(f"Loaded {RESULTS_PATH.name}")
print(f"Top-level keys: {sorted(data.keys())}")
print(f"Study guid:     {data.get('study_guid')}")
print(f"Sample summary: {data.get('sample', {}).get('participants_completed')} completed "
      f"/ {data.get('sample', {}).get('participants_total')} total")

Loaded results_JsbnDzlvW7kWgGngu0fMwQkaQY45ozgC-2.json
Top-level keys: ['approaches', 'insights', 'moderators', 'participants', 'participants_table', 'questionnaire', 'sample', 'study_guid']
Study guid:     JsbnDzlvW7kWgGngu0fMwQkaQY45ozgC
Sample summary: 2 completed / 2 total


## Optional – time filter

Drop pre-launch / dev test runs by keeping only participations that **finished**
within a window.  Set `SINCE` and/or `UNTIL` to `None` to disable a bound.
In-progress participations (where `time_finished is None`) are dropped whenever
`SINCE` is set, since we can't tell if they belong to the window yet.

In [19]:
# ISO-8601 (UTC) strings.  Participation `time_finished` values use the same
# format so a lexicographic ">=" / "<=" comparison is safe.  Set either bound
# to None to disable it; set both to None to keep everything.
SINCE = "2026-04-17T12:00:00"
UNTIL = None


def _in_window(ts: str | None) -> bool:
    if ts is None:
        return SINCE is None
    if SINCE is not None and ts < SINCE:
        return False
    if UNTIL is not None and ts > UNTIL:
        return False
    return True


before_p = len(data.get("participants", []))
before_t = len(data.get("participants_table", []))

data["participants"] = [p for p in data.get("participants", []) if _in_window(p.get("time_finished"))]
data["participants_table"] = [r for r in data.get("participants_table", []) if _in_window(r.get("time_finished"))]

print(f"participants:        {before_p} -> {len(data['participants'])}")
print(f"participants_table:  {before_t} -> {len(data['participants_table'])}")
print(f"window: SINCE={SINCE!r}  UNTIL={UNTIL!r}")

participants:        2 -> 2
participants_table:  2 -> 2
window: SINCE='2026-04-17T12:00:00'  UNTIL=None


## 1. `participants_table` → DataFrame

This is the exact list the admin UI renders in the *Participants* tab.
Each row ↔ one `Participation`, with Prolific IDs, duration, attention-check
outcome and the approach order (numeric + named) already joined in.

In [16]:
def _flatten_table_row(row: dict) -> dict:
    prolific = row.get("prolific") or {}
    attention = row.get("attention_checks") or {}
    approach = row.get("approach_order") or []
    effective = row.get("effective_order") or []

    flat = {
        "participation_id":    row.get("participation_id"),
        "uuid":                row.get("uuid"),
        "status":              row.get("status"),
        "time_joined":         row.get("time_joined"),
        "time_finished":       row.get("time_finished"),
        "duration_sec":        row.get("duration_sec"),
        "search_events_count": row.get("search_events_count"),
        "prolific_pid":        prolific.get("pid"),
        "prolific_study":      prolific.get("study_id"),
        "prolific_session":    prolific.get("session_id"),
        "attention_overall_pass": attention.get("overall_pass"),
        "attention_phase_passed": attention.get("phase_passed_count"),
        "attention_phase_total":  attention.get("phase_total_count"),
        "attention_final_pass":   attention.get("final_pass"),
        "approach_order":   "/".join(str(x) for x in approach) if approach else None,
        "effective_order":  " → ".join(effective) if effective else None,
    }

    # One column per individual phase attention check (phase_1_pass, phase_2_pass, …).
    for entry in attention.get("phase_passes") or []:
        phase = entry.get("phase")
        if phase is None:
            continue
        flat[f"attention_phase_{int(phase) + 1}_pass"] = entry.get("passed")

    return flat


participants_df = pd.DataFrame([_flatten_table_row(r) for r in data.get("participants_table", [])])
print(f"participants_df: {participants_df.shape[0]} rows × {participants_df.shape[1]} cols")
participants_df.head()

participants_df: 0 rows × 0 cols


""


## 2. Movie-feedback events → long DataFrame

One row per (participation, phase, iteration, movie, rating) tuple.

In [17]:
feedback_rows = []
for participant in data.get("participants", []):
    pid = participant.get("participation_id")
    pp_id = participant.get("participant_id")
    for phase_idx, phase in enumerate(participant.get("phase_movie_feedback", []) or []):
        for event in phase.get("events", []) or []:
            feedback_rows.append({
                "participation_id": pid,
                "participant_id":   pp_id,
                "phase":            phase.get("phase", phase_idx),
                "approach_name":    phase.get("approach_name"),
                "iteration":        event.get("iteration"),
                "movie_id":         event.get("movie_id"),
                "rating":           event.get("rating"),
                "time":             event.get("time"),
            })

feedback_df = pd.DataFrame(feedback_rows)
print(f"feedback_df: {feedback_df.shape[0]} rows × {feedback_df.shape[1]} cols")
feedback_df.head()

feedback_df: 0 rows × 0 cols


""


## 3. Phase + final questionnaires → wide DataFrame

One row per (participation, phase), with the per-phase Likert answers plus
the final questionnaire columns prefixed by `final__`.

In [ ]:
questionnaire_rows = []
for participant in data.get("participants", []):
    pid = participant.get("participation_id")
    pp_id = participant.get("participant_id")
    final = {f"final__{k}": v for k, v in (participant.get("final_questionnaire") or {}).items()}
    for phase_idx, phase_q in enumerate(participant.get("phase_questionnaires", []) or []):
        row = {
            "participation_id": pid,
            "participant_id":   pp_id,
            "phase":            phase_q.get("phase", phase_idx),
            "approach_name":    phase_q.get("approach_name"),
            **(phase_q.get("answers") or {}),
            **final,
        }
        questionnaire_rows.append(row)

questionnaire_df = pd.DataFrame(questionnaire_rows)
print(f"questionnaire_df: {questionnaire_df.shape[0]} rows × {questionnaire_df.shape[1]} cols")
questionnaire_df.head()

## 4. Quick sanity checks

In [ ]:
print("Status counts:")
print(participants_df["status"].value_counts(dropna=False).to_string())

# Attention checks (three per completed participant: two phase + one final).
attn_cols = [c for c in participants_df.columns if c.startswith("attention_")]
print("\nAttention columns:", attn_cols)
for col in ["attention_overall_pass", "attention_final_pass"] + [c for c in participants_df.columns if c.startswith("attention_phase_") and c.endswith("_pass")]:
    if col not in participants_df:
        continue
    s = participants_df[col]
    passed = int((s == True).sum())
    failed = int((s == False).sum())
    unknown = int(s.isna().sum())
    print(f"  {col:35s} pass={passed}  fail={failed}  unknown={unknown}")

print("\nApproach order (numeric):")
print(participants_df["approach_order"].value_counts(dropna=False).to_string())

print("\nApproach order (named):")
print(participants_df["effective_order"].value_counts(dropna=False).to_string())

print("\nDuration (seconds):")
print(participants_df["duration_sec"].describe().to_string())

In [ ]:
if not feedback_df.empty:
    per_phase_likes = (
        feedback_df[feedback_df["rating"].astype(str).str.lower() == "like"]
        .groupby(["participation_id", "phase"])["movie_id"]
        .nunique()
        .unstack("phase")
        .fillna(0)
        .astype(int)
    )
    per_phase_likes.columns = [f"phase_{c}_likes" for c in per_phase_likes.columns]
    display(per_phase_likes.head())
else:
    print("feedback_df is empty – no phase_movie_feedback events in export.")

## 5. Paying participants — payout table

Only completed participations with a Prolific PID and a passed attention
check are marked `pay=True`.  The remaining rows need a manual decision
(approve / reject / reach out).

In [ ]:
payout_cols = [
    "participation_id",
    "uuid",
    "prolific_pid",
    "prolific_study",
    "prolific_session",
    "status",
    "duration_sec",
    "attention_overall_pass",
    "attention_phase_passed",
    "attention_phase_total",
    "attention_final_pass",
    "effective_order",
]
payout = participants_df[[c for c in payout_cols if c in participants_df.columns]].copy()

payout["pay"] = (
    payout["status"].eq("completed")
    & payout["prolific_pid"].notna()
    & payout["attention_overall_pass"].eq(True)
)

print(f"To pay: {int(payout['pay'].sum())} / {len(payout)}")
payout.sort_values(["pay"], ascending=[False])
